In [40]:
import requests
import re
import json
from Bio import SeqIO
import subprocess
import sys

class MyFastaParser:
    def __init__(self, file_name):
        self.filename = file_name

    def _get_uniprot(self, accession):
        '''
        define http_function to get the data from Uniprot API
        '''
        endpoint = f"https://rest.uniprot.org/uniprotkb/{accession}"

        http_args = {
            "params": {"format": "json"} 
        }
        def http_function(url, **kwargs):
            response = requests.get(url, **kwargs)
            if response.status_code == 200:
                return response.json()
            else:
                print(f"Error {response.status_code} ")
                return None
        resp_uniprot =http_function(endpoint, **http_args)
        return resp_uniprot
        

    def _get_ensembl(self, id):
        '''
        define http_function to get the data from Ensembl API
        '''
        endpoint = f'https://rest.ensembl.org/lookup/id/{id}'
        
        http_args = {"headers": {"Content-Type" : "application/json"}}
        def http_function(url, **kwargs):
            response = requests.get(url, **kwargs)
            if response.status_code == 200:
                return response.json()
            else:
                print(f"Error {response.status_code} ")
                return None
        resp_ens = http_function(endpoint, **http_args)
        return resp_ens
    


    def _uniprot_parse_response(self, resp_uniprot: dict):
        '''
        parse response from Uniprot and output
        organism, geneInfo, sequenceInfo, type

        do not forget to include error handling
        '''

        output = {}

        try:
            output["organism"] = resp_uniprot.get("organism", {}).get("scientificName", None)

            genes = resp_uniprot.get("genes", [])
            if genes:
                output["geneInfo"] = genes[0]
            else:
                output["geneInfo"] = None

            output["sequenceInfo"] = resp_uniprot.get("sequence", {})

            output["type"] = "protein"


        except Exception as e:
            print("Error parsing response:", e)
            return None

        return output


    def _ensembl_parse_response(self, resp_ens: dict):
        '''
        parse Ensembl response and output
        object_type, assembly_name, species, db_type, biotype, display_name, id, description, canonical_transcript, source

        do not forget to include error handling (e.g. for key errors)
        '''
        output = {}

        try:
            # organism
            output['object_type'] = resp_ens['object_type']
            output['assembly_name'] = resp_ens['assembly_name']
            output['species'] = resp_ens['species']
            output['db_type'] = resp_ens['db_type']
            output['biotype'] = resp_ens['biotype']
            output['db_type'] = resp_ens['db_type']
            output['display_name'] = resp_ens['display_name']
            output['id'] = resp_ens['id']
            output['description'] = resp_ens['description']
            output['canonical_transcript'] = resp_ens['canonical_transcript']
            output['source'] = resp_ens['source']
        except Exception as e:
            print("Error parsing response:", e)
            return None

        return output

    def _access_database(self, id, database, seq_description, seq_sequence) -> dict:
        # this function calls either _get_uniprot() or _get_ensembl()
        # an then _uniprot_parse_response() or _ensembl_parse_response()
        # outputs a dictionary with results (see example in test_data)
        output = {}
        if database.lower() == "uniprot":
            resp = self._get_uniprot(id)
            if resp:
                db_info = self._uniprot_parse_response(resp)
                return {"DB_name": "uniprot",
                    "description": seq_description,
                    "sequence": seq_sequence,
                    "database_info": db_info}
            else:
                db_info = None
        elif database.lower() == "ensembl":
            resp = self._get_ensembl(id)
            if resp:
                db_info = self._ensembl_parse_response(resp)
                return {"DB_name": "ensembl",
                    "description": seq_description,
                    "sequence": seq_sequence,
                    "database_info": db_info}
            else:
                db_info = None
        else:
            raise ValueError(f"Unknown database: {database}")
        
        
        return output

    def seqkit_stats(self) -> dict:
        try:
            seqkit = subprocess.run(
                ["seqkit", "stats", self.filename, "-a"],  
                capture_output=True,
                text=True,
                check=True
            )
            seqkit_out = seqkit.stdout.strip().split('\n')

            # split names and values
            prop_names = seqkit_out[0].split()[1:]
            prop_vals = seqkit_out[1].split()[1:]

            seqkit_result = dict(zip(prop_names, prop_vals))
            fasta_type = seqkit_result.get("type", "Unknown")
            fasta_num_seqs = int(seqkit_result.get("num_seqs", 0))
            return {
            "fasta_seqkit_stat_info": seqkit_result,
            "fasta_type": fasta_type,
            "fasta_num_seqs": fasta_num_seqs
        }
    
        
        except subprocess.CalledProcessError as e:
        # ошибка самого seqkit
            raise RuntimeError(f"seqkit failed:\n{e.stderr}")

        except Exception as e:
            raise RuntimeError(f"Parsing error: {e}")
       
            
        # this function calls seqkit via subprocess
        # if error arises, returns stderr and finishes the parser execution
        # if stats are collected, return the result (see example in test_data)

        

    def biopython_parser(self, seqkit_result) -> dict:
        output = {}

        fasta_type = seqkit_result["fasta_type"].lower()

        if fasta_type == "protein":
            regex = r"(?:sp|tr)\|([A-Z0-9]+)\|"
            source = "uniprot"

        elif fasta_type == "dna":
            regex = r"^(ENS[A-Z]*\d+)"
            source = "ensembl"

        else:
            raise ValueError("Unknown FASTA type")

        for record in SeqIO.parse(self.filename, "fasta"):
            header = record.description
            sequence = str(record.seq)

            match = re.search(regex, header)
            seq_id = match.group(1) if match else None

            if seq_id:
                db_info = self._access_database(seq_id, source, header, sequence)
            else:
                db_info = {"warning": "No ID match found"}

            entry = {
                "DB_name": source,
                "description": header,
                "sequence": sequence,
                "database_info": db_info
            }

            output[seq_id if seq_id else header] = entry

        return output

    def show_output(self, output, indent=0):
        for key, value in output.items():
            print('\t' * indent + str(key))
            if isinstance(value, dict):
                self.show_output(value, indent + 1)
            else:
                print('\t' * (indent + 1) + str(value))

In [59]:
parser = MyFastaParser('test_file.fasta')

In [60]:
stats = parser.seqkit_stats()

In [61]:
stats

{'fasta_seqkit_stat_info': {'format': 'FASTA',
  'type': 'Protein',
  'num_seqs': '2',
  'sum_len': '456',
  'min_len': '29',
  'avg_len': '228',
  'max_len': '427',
  'Q1': '29',
  'Q2': '228',
  'Q3': '427',
  'sum_gap': '0',
  'N50': '427',
  'N50_num': '1',
  'Q20(%)': '0',
  'Q30(%)': '0',
  'AvgQual': '0',
  'GC(%)': '0',
  'sum_n': '0'},
 'fasta_type': 'Protein',
 'fasta_num_seqs': 2}

In [62]:
biopython = parser.biopython_parser(stats)

parser.show_output(biopython)

P11473
	DB_name
		uniprot
	description
		sp|P11473|VDR_HUMAN Vitamin D3 receptor OS=Homo sapiens OX=9606 GN=VDR PE=1 SV=1
	sequence
		MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFHFNAMTCEGCKGFFRRSMKRKALFTCPFNGDCRITKDNRRHCQACRLKRCVDIGMMKEFILTDEEVQRKREMILKRKEEEALKDSLRPKLSEEQQRIIAILLDAHHKTYDPTYSDFCQFRPPVRVNDGGGSHPSRPNSRHTPSFSGDSSSSCSDHCITSSDMMDSSSFSNLDLSEEDSDDPSVTLELSQLSMLPHLADLVSYSIQKVIGFAKMIPGFRDLTSEDQIVLLKSSAIEVIMLRSNESFTMDDMSWTCGNQDYKYRVSDVTKAGHSLELIEPLIKFQVGLKKLNLHEEEHVLLMAICIVSPDRPGVQDAALIEAIQDRLSNTLQTYIRCRHPPPGSHLLYAKMIQKLADLRSLNEEHSKQYRCLSFQPECSMKLTPLVLEVFGNEIS
	database_info
		DB_name
			uniprot
		description
			sp|P11473|VDR_HUMAN Vitamin D3 receptor OS=Homo sapiens OX=9606 GN=VDR PE=1 SV=1
		sequence
			MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFHFNAMTCEGCKGFFRRSMKRKALFTCPFNGDCRITKDNRRHCQACRLKRCVDIGMMKEFILTDEEVQRKREMILKRKEEEALKDSLRPKLSEEQQRIIAILLDAHHKTYDPTYSDFCQFRPPVRVNDGGGSHPSRPNSRHTPSFSGDSSSSCSDHCITSSDMMDSSSFSNLDLSEEDSDDPSVTLELSQLSMLPHLADLVSYSIQKVIGFAKMIPGFRDLTSEDQIVLLKSSAIEVIMLRSNESFTMDDMSWTCGN

In [63]:
parser_1 = MyFastaParser('ensembl_download_1.fasta')


In [64]:
stats_1 = parser_1.seqkit_stats()
print(stats_1)

{'fasta_seqkit_stat_info': {'format': 'FASTA', 'type': 'DNA', 'num_seqs': '6', 'sum_len': '86', 'min_len': '9', 'avg_len': '14.3', 'max_len': '23', 'Q1': '10', 'Q2': '13.5', 'Q3': '17', 'sum_gap': '0', 'N50': '16', 'N50_num': '3', 'Q20(%)': '0', 'Q30(%)': '0', 'AvgQual': '0', 'GC(%)': '45.35', 'sum_n': '0'}, 'fasta_type': 'DNA', 'fasta_num_seqs': 6}


In [65]:
biopython_1 = parser_1.biopython_parser(stats_1)
parser_1.show_output(biopython_1)

Error parsing response: 'description'
Error parsing response: 'description'
Error parsing response: 'description'
Error parsing response: 'description'
Error 400 
Error parsing response: 'display_name'
ENSMUST00000196221
	DB_name
		ensembl
	description
		ENSMUST00000196221.2 cds chromosome:GRCm39:14:54350925:54350933:1 gene:ENSMUSG00000096749.3 gene_biotype:TR_D_gene transcript_biotype:TR_D_gene gene_symbol:Trdd1 description:T cell receptor delta diversity 1 [Source:MGI Symbol;Acc:MGI:4439547]
	sequence
		ATGGCATAT
	database_info
		DB_name
			ensembl
		description
			ENSMUST00000196221.2 cds chromosome:GRCm39:14:54350925:54350933:1 gene:ENSMUSG00000096749.3 gene_biotype:TR_D_gene transcript_biotype:TR_D_gene gene_symbol:Trdd1 description:T cell receptor delta diversity 1 [Source:MGI Symbol;Acc:MGI:4439547]
		sequence
			ATGGCATAT
		database_info
			None
ENSMUST00000177564
	DB_name
		ensembl
	description
		ENSMUST00000177564.2 cds chromosome:GRCm39:14:54359683:54359698:1 gene:ENSMUSG000

In [66]:
parser_2 = MyFastaParser('ensembl_download_2.fasta')

In [67]:
stats_2 = parser_2.seqkit_stats()
print(stats_2)

RuntimeError: Parsing error: list index out of range

In [68]:
parser_3 = MyFastaParser('uniprot_download.fasta')

In [69]:
stats_3 = parser_3.seqkit_stats()
print(stats_3)

{'fasta_seqkit_stat_info': {'format': 'FASTA', 'type': 'Protein', 'num_seqs': '7', 'sum_len': '3,861', 'min_len': '180', 'avg_len': '551.6', 'max_len': '1,382', 'Q1': '429', 'Q2': '441', 'Q3': '500', 'sum_gap': '0', 'N50': '468', 'N50_num': '3', 'Q20(%)': '0', 'Q30(%)': '0', 'AvgQual': '0', 'GC(%)': '0', 'sum_n': '0'}, 'fasta_type': 'Protein', 'fasta_num_seqs': 7}


In [70]:
biopython_3 = parser_3.biopython_parser(stats_3)
parser_3.show_output(biopython_3)

Q9R1A7
	DB_name
		uniprot
	description
		sp|Q9R1A7|NR1I2_RAT Nuclear receptor subfamily 1 group I member 2 OS=Rattus norvegicus OX=10116 GN=Nr1i2 PE=2 SV=1
	sequence
		MRPEERWNHVGLVQREEADSVLEEPINVDEEDGGLQICRVCGDKANGYHFNVMTCEGCKGFFRRAMKRNVRLRCPFRKGTCEITRKTRRQCQACRLRKCLESGMKKEMIMSDAAVEQRRALIKRKKREKIEAPPPGGQGLTEEQQALIQELMDAQMQTFDTTFSHFKDFRLPAVFHSDCELPEVLQASLLEDPATWSQIMKDSVPMKISVQLRGEDGSIWNYQPPSKSDGKEIIPLLPHLADVSTYMFKGVINFAKVISHFRELPIEDQISLLKGATFEMCILRFNTMFDTETGTWECGRLAYCFEDPNGGFQKLLLDPLMKFHCMLKKLQLREEEYVLMQAISLFSPDRPGVVQRSVVDQLQERFALTLKAYIECSRPYPAHRFLFLKIMAVLTELRSINAQQTQQLLRIQDTHPFATPLMQELFSSTDG
	database_info
		DB_name
			uniprot
		description
			sp|Q9R1A7|NR1I2_RAT Nuclear receptor subfamily 1 group I member 2 OS=Rattus norvegicus OX=10116 GN=Nr1i2 PE=2 SV=1
		sequence
			MRPEERWNHVGLVQREEADSVLEEPINVDEEDGGLQICRVCGDKANGYHFNVMTCEGCKGFFRRAMKRNVRLRCPFRKGTCEITRKTRRQCQACRLRKCLESGMKKEMIMSDAAVEQRRALIKRKKREKIEAPPPGGQGLTEEQQALIQELMDAQMQTFDTTFSHFKDFRLPAVFHSDCELPEVLQASLLEDPATWSQIMKDSVPMKISVQLRGEDGS